#  MLflow — Tracking des expériences

**Objectif :** Enregistrer toutes les expériences du projet
pour assurer la traçabilité et la reproductibilité.

**Backend :** SQLite local (standard entreprise pour projets individuels)
**Artifact store :** Dossier local mlruns/artifacts

In [1]:
import os
import time
import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # ← backend non-interactif, évite les conflits
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.sklearn

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    confusion_matrix, roc_auc_score,
    average_precision_score, f1_score,
    recall_score, precision_score
)

print(f" Imports OK — MLflow {mlflow.__version__}")

 Imports OK — MLflow 3.14.0


In [2]:
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()

print(f" Données chargées")
print(f"   Train : {X_train.shape[0]:,} lignes")
print(f"   Test  : {X_test.shape[0]:,} lignes")
print(f"   Fraudes test : {(y_test==1).sum()}")

 Données chargées
   Train : 454,902 lignes
   Test  : 56,962 lignes
   Fraudes test : 98


In [ ]:
import os, mlflow

DB_PATH      = os.path.abspath('../mlruns/mlflow.db')
ARTIFACT_DIR = os.path.abspath('../mlruns/artifacts')

os.makedirs('../mlruns', exist_ok=True)
os.makedirs(ARTIFACT_DIR, exist_ok=True)

artifact_uri = 'file:///' + ARTIFACT_DIR.replace('\\', '/').lstrip('/')
tracking_uri = 'sqlite:///' + DB_PATH.replace('\\', '/')

mlflow.set_tracking_uri(tracking_uri)

EXP_NAME = "fraud-detection-v1"

exp = mlflow.get_experiment_by_name(EXP_NAME)
if exp is None:
    mlflow.create_experiment(name=EXP_NAME, artifact_location=artifact_uri)

mlflow.set_experiment(EXP_NAME)

print(f" Connecté à la DB MLflow 2.17.2")
print(f"   Tracking URI : {tracking_uri}")
print(f"   Artifact URI : {artifact_uri}")

 Expérience créée — ID : 2
   Tracking URI  : sqlite:///c:/Users/Eartrh/Documents/Etudes_France_2025/Science_De_Données/fraud-detection-ml/mlruns/mlflow.db
   Artifact URI  : file:///c:/Users/Eartrh/Documents/Etudes_France_2025/Science_De_Données/fraud-detection-ml/mlruns/artifacts


In [7]:
def run_experiment(name, model, params, threshold=0.5, role="candidate"):
    """
    Entraîne un modèle, calcule les métriques,
    logue tout dans MLflow. Retourne (run_id, metrics).
    """
    with mlflow.start_run(run_name=name, experiment_id=mlflow.get_experiment_by_name(EXP_NAME).experiment_id):

        # Tags
        mlflow.set_tag("model",   name)
        mlflow.set_tag("author",  "TinhinaneBA")
        mlflow.set_tag("role",    role)
        mlflow.set_tag("dataset", "creditcard_fraud")

        # Paramètres
        mlflow.log_param("threshold",  threshold)
        mlflow.log_param("smote",      "yes")
        mlflow.log_param("scaler",     "RobustScaler")
        mlflow.log_param("n_features", X_train.shape[1])
        for k, v in params.items():
            mlflow.log_param(k, v)

        # Entraînement
        t0 = time.time()
        model.fit(X_train, y_train)
        duration = round(time.time() - t0, 1)
        mlflow.log_metric("train_seconds", duration)

        # Prédictions
        y_proba = model.predict_proba(X_test)[:, 1]
        y_pred  = (y_proba >= threshold).astype(int)

        # Métriques
        cm             = confusion_matrix(y_test, y_pred)
        tn, fp, fn, tp = cm.ravel()

        auc_roc = round(roc_auc_score(y_test, y_proba), 4)
        pr_auc  = round(average_precision_score(y_test, y_proba), 4)
        f1      = round(f1_score(y_test, y_pred), 4)
        recall  = round(recall_score(y_test, y_pred), 4)
        prec    = round(precision_score(y_test, y_pred, zero_division=0), 4)

        mlflow.log_metric("auc_roc",         auc_roc)
        mlflow.log_metric("pr_auc",          pr_auc)
        mlflow.log_metric("f1_score",        f1)
        mlflow.log_metric("recall_fraud",    recall)
        mlflow.log_metric("precision",       prec)
        mlflow.log_metric("true_positives",  int(tp))
        mlflow.log_metric("false_negatives", int(fn))
        mlflow.log_metric("false_positives", int(fp))
        mlflow.log_metric("true_negatives",  int(tn))

        # Modèle — trusted types pour LightGBM + XGBoost
        TRUSTED_TYPES = [
            "collections.OrderedDict",
            "lightgbm.basic.Booster",
            "lightgbm.sklearn.LGBMClassifier",
            "xgboost.sklearn.XGBClassifier",
            "xgboost.core.Booster",
            "sklearn.linear_model._logistic.LogisticRegression",
            "sklearn.ensemble._forest.RandomForestClassifier",
            "numpy.dtype",
            "numpy.ndarray",
        ]
        mlflow.sklearn.log_model(
            model,
            name                = "model",
            skops_trusted_types = TRUSTED_TYPES
        )

        # ← Ces deux lignes étaient manquantes dans ta version
        run_id  = mlflow.active_run().info.run_id

    # ← En dehors du with — après fermeture du run
    metrics = {
        "auc_roc": auc_roc, "pr_auc": pr_auc,
        "f1": f1, "recall": recall, "precision": prec,
        "tp": tp, "fn": fn, "fp": fp, "tn": tn,
        "train_seconds": duration
    }

    print(f" {name:35s} PR-AUC={pr_auc:.4f} "
          f"Recall={recall:.4f} F1={f1:.4f} ({duration}s)")

    return run_id, metrics

print(" Fonction run_experiment définie")

 Fonction run_experiment définie


In [8]:
results = {}

p_lr = {"max_iter": 1000, "class_weight": "balanced", "random_state": 42}
results['LogReg'] = run_experiment(
    name      = "LogisticRegression",
    model     = LogisticRegression(**p_lr),
    params    = p_lr,
    threshold = 0.5,
    role      = "baseline"
)

2026/07/14 09:41:22 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\Eartrh\AppData\Local\Temp\tmp2hj6cwda\model\model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.9.0', 'skops==0.14.0']. Set logging level to DEBUG to see the full traceback. 


 LogisticRegression                  PR-AUC=0.7235 Recall=0.9184 F1=0.1110 (31.8s)


In [9]:
p_lgbm = {
    "n_estimators": 200, "learning_rate": 0.1,
    "max_depth": 6, "random_state": 42, "verbose": -1
}
results['LightGBM'] = run_experiment(
    name      = "LightGBM",
    model     = LGBMClassifier(**p_lgbm),
    params    = p_lgbm,
    threshold = 0.5,
    role      = "candidate"
)

2026/07/14 09:45:51 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\Eartrh\AppData\Local\Temp\tmphb8fcwxx\model\model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.9.0', 'skops==0.14.0']. Set logging level to DEBUG to see the full traceback. 


 LightGBM                            PR-AUC=0.8566 Recall=0.8673 F1=0.7143 (128.1s)


In [10]:
p_xgb = {
    "n_estimators": 200, "learning_rate": 0.1,
    "max_depth": 6, "eval_metric": "aucpr",
    "random_state": 42, "verbosity": 0
}
results['XGB_05'] = run_experiment(
    name      = "XGBoost_threshold_0.50",
    model     = XGBClassifier(**p_xgb),
    params    = p_xgb,
    threshold = 0.5,
    role      = "candidate"
)

 XGBoost_threshold_0.50              PR-AUC=0.8644 Recall=0.8571 F1=0.7179 (239.1s)


In [11]:
with open('../models/model_metadata.json') as f:
    meta = json.load(f)

THRESHOLD = meta['threshold']
print(f"Seuil optimal chargé : {THRESHOLD}")

results['XGB_PROD'] = run_experiment(
    name      = "XGBoost_PRODUCTION",
    model     = XGBClassifier(**p_xgb),
    params    = p_xgb,
    threshold = THRESHOLD,
    role      = "PRODUCTION"
)

Seuil optimal chargé : 0.9400000000000001


2026/07/14 10:21:07 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\Eartrh\AppData\Local\Temp\tmpgicdliys\model\model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.9.0', 'skops==0.14.0']. Set logging level to DEBUG to see the full traceback. 


 XGBoost_PRODUCTION                  PR-AUC=0.8644 Recall=0.8163 F1=0.8649 (244.9s)


In [12]:
rows = []
for name, (run_id, m) in results.items():
    rows.append({
        'Modèle'        : name,
        'PR-AUC'        : m['pr_auc'],
        'Recall Fraude' : m['recall'],
        'F1-Score'      : m['f1'],
        'Précision'     : m['precision'],
        'VP'            : m['tp'],
        'FN'            : m['fn'],
        'FP'            : m['fp'],
        'Temps (s)'     : m['train_seconds']
    })

df_results = pd.DataFrame(rows).set_index('Modèle')

print("═" * 80)
print("  TABLEAU COMPARATIF — TOUS LES RUNS")
print("═" * 80)
print(df_results.to_string())
print("═" * 80)

# ── Visualisation ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics_plot = ['PR-AUC', 'Recall Fraude', 'F1-Score']
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for ax, metric in zip(axes, metrics_plot):
    vals  = df_results[metric].values
    names = df_results.index.tolist()
    bars  = ax.barh(names, vals, color=colors[:len(names)],
                    edgecolor='white', height=0.5)
    ax.set_title(metric, fontweight='bold')
    ax.set_xlim([0, 1.1])
    for bar, val in zip(bars, vals):
        ax.text(val + 0.01,
                bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=9)
    ax.tick_params(axis='y', labelsize=9)

plt.suptitle(' MLflow — Comparaison des runs',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/17_mlflow_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print(" Figure sauvegardée → reports/figures/17_mlflow_comparison.png")

════════════════════════════════════════════════════════════════════════════════
  TABLEAU COMPARATIF — TOUS LES RUNS
════════════════════════════════════════════════════════════════════════════════
          PR-AUC  Recall Fraude  F1-Score  Précision  VP  FN    FP  Temps (s)
Modèle                                                                       
LogReg    0.7235         0.9184    0.1110     0.0591  90   8  1434       31.8
LightGBM  0.8566         0.8673    0.7143     0.6071  85  13    55      128.1
XGB_05    0.8644         0.8571    0.7179     0.6176  84  14    52      239.1
XGB_PROD  0.8644         0.8163    0.8649     0.9195  80  18     7      244.9
════════════════════════════════════════════════════════════════════════════════
 Figure sauvegardée → reports/figures/17_mlflow_comparison.png


In [13]:
db_uri = 'sqlite:///' + DB_PATH.replace('\\', '/')
print(f"""
╔══════════════════════════════════════════════════════════════╗
║              VISUALISER LES RUNS — MLFLOW UI                 ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Ouvre un NOUVEAU terminal PowerShell et tape :              ║
║                                                              ║
║  cd fraud-detection-ml                                       ║
║  venv\\Scripts\\Activate.ps1                                  ║
║  mlflow ui --backend-store-uri "{db_uri}"
║                                                              ║
║  Puis ouvre : http://127.0.0.1:5000                          ║
║                                                              ║
║  Tu verras tous les runs, métriques et paramètres.           ║
╚══════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════╗
║              VISUALISER LES RUNS — MLFLOW UI                 ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Ouvre un NOUVEAU terminal PowerShell et tape :              ║
║                                                              ║
║  cd fraud-detection-ml                                       ║
║  venv\Scripts\Activate.ps1                                  ║
║  mlflow ui --backend-store-uri "sqlite:///c:/Users/Eartrh/Documents/Etudes_France_2025/Science_De_Données/fraud-detection-ml/mlruns/mlflow.db"
║                                                              ║
║  Puis ouvre : http://127.0.0.1:5000                          ║
║                                                              ║
║  Tu verras tous les runs, métriques et paramètres.           ║
╚══════════════════════════════════════════════════════════════╝



##  Synthèse MLflow

### Runs enregistrés
| Run | Modèle | Rôle |
|---|---|---|
| 1 | Logistic Regression | Baseline |
| 2 | LightGBM | Candidat |
| 3 | XGBoost seuil 0.50 | Candidat |
| 4 | XGBoost seuil 0.94 | **PRODUCTION** |

> Random Forest exclu volontairement —
> temps d'entraînement (~10h) inexploitable en production.
> Décision documentée dans 03_modeling.ipynb.

### Valeur professionnelle
- Chaque run est horodaté et identifié par un run_id unique
- Paramètres, métriques et modèle sauvegardés automatiquement
- Reproductibilité garantie — on peut recréer n'importe quel run
- Audit trail complet — exigence réglementaire en finance

### Prochaine étape → app/streamlit_app.py